In [2]:
import dreams_python
from dreams_python import emulation

In [8]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import optuna

# Hyperparameter optimization of models

## see also Tutorial 10. Normalizing Flows

In this tutorial, we're going to create a hyperparameter optimized NeHOD normalizing flows emulator.

There are a ton of hyperparameters associated with these models, the ones that we're going to be tuning in this notebook are:

- `num_transforms`: number of transforms to do
- `projection_dim`: number of nodes in projection dimension
- `hidden_dim`: number of nodes in hidden layers
- `n_projection_layers`: number of projection layers
- `n_hidden_layers`: number of hidden layers
- `dropout`: fraction of nodes to drop
- `lr`: learning rate
- `decay`: weight decay
- `batch_size`: size of batches

## but first, load in our data

In [4]:
data_file = '/home/aku7cf/DREAMS/mass_varied/calibration/data/sim_data.hdf5'

## load in simulation parameters 
mvs = dreams_python.DREAMS('/standard/DREAMS',suite='varied_mass',DM_type='CDM',sobol_number=6)
params, header = mvs.read_param_file('TNG_SB6.txt')

In [5]:
## load in (and clean) my data
with h5py.File(data_file,'r') as f:
    halo_mass    = np.log10(f['HaloMass'][...]+1e5)
    stellar_mass = np.log10(f['StellarMass'][...]+1e3)
    bh_mass      = np.log10(f['BHMass'][...]+1e3)
    gas_mass     = np.log10(f['GasMass'][...]+1e3)
    sizes        = np.log10(f['Size'][...]+1e-4)
    gas_metals   = f['GasMetallicity'][...]
    sfrs         = np.log10(f['SFRs'][...]+1e-4)

_labels_raw = np.column_stack([
    halo_mass, stellar_mass, bh_mass, gas_mass, sizes, gas_metals, sfrs
])
_finite_mask = np.all(np.isfinite(_labels_raw), axis=1)
n_bad = (~_finite_mask).sum()
if n_bad > 0:
    print(f"Dropping {n_bad}/{len(_finite_mask)} simulations with NaN/inf values.")

halo_mass    = halo_mass[_finite_mask]
stellar_mass = stellar_mass[_finite_mask]
bh_mass      = bh_mass[_finite_mask]
gas_mass     = gas_mass[_finite_mask]
sizes        = sizes[_finite_mask]
gas_metals   = gas_metals[_finite_mask]
sfrs         = sfrs[_finite_mask]
params       = params[_finite_mask]  ## keep params in sync!!

features = np.array([
    params[:, 0],
    params[:, 1],
    params[:, 2],
    np.log10(params[:, 3]), ## sn1, sn2, and AGN are log spaced
    np.log10(params[:, 4]),
    np.log10(params[:, 5])
]).T
labels = np.column_stack([
    halo_mass   ,
    stellar_mass,
    bh_mass     ,
    gas_mass    ,
    sizes       ,
    gas_metals  ,
    sfrs        
])

Dropping 396/1024 simulations with NaN/inf values.


## Do hyperparameter tuning

In [ ]:
## feed in simulation parameters (features) and data (labels) 
##   name your emulator (here it is "my_test")
##   point to your data directory (in my case I have it saved in a direct)
name = 'hyperparameter_search'
flow = emulator(features, labels, name) ## feed in simulation parameters (features) and data (labels)

## tune the model (takes some time) with 30 trials each having 5000 steps
#    Note that these values are reasonable for my dataset, but might not be for yours!
flow.tune_flow(n_trials=30,max_steps=5000)

## With best hyperparameters, train a model

In [ ]:
## load study from optuna (directory structure might be different if you set it differently)
study = optuna.load_study(
    study_name=f"{name}_tuning",
    storage=f"sqlite:///emulator/flows/{name}/optuna/optuna.db",
)

## get best parameters
best_params = study.best_params

## train your emulator based on the best parameters
flow.train_flow(
    **{k: best_params[k] for k in [
        'num_transforms','projection_dim','hidden_dim',
        'n_projection_layers','n_hidden_layers','dropout','batch_size'
    ]},
    lr=best_params['lr'],
    decay=best_params['wd'],
    seed=1234,
    max_steps=5000 ## same as tuning
)

## Validate loss function

In [ ]:
(train_loss, train_epoch), (valid_loss, valid_epoch) = flow.loss_data()

fig = plt.figure()
plt.plot(train_epoch, train_loss, label='Train')
plt.plot(valid_epoch, valid_loss, label='Validation')
plt.legend(frameon=False)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

## Make predictions from your model

In [ ]:
context = np.column_stack([ ## give same parameters in
    halo_mass   ,
    params[:, 1],
    params[:, 2],
    np.log10(params[:, 3]),
    np.log10(params[:, 4]),
    np.log10(params[:, 5])
])

posterior_draws = flow.make_prediction(context, n_samples=1000)
## the more samples you have the more converged different models will be

prediction = posterior_draws.mean(axis=0)
aleatoric_uncertainty = posterior_draws.std(axis=0)

## Once you have multiple models done...

You should probably not just hyperparameter optimize 1 model, but do a bunch ($\sim10-100$?) and pick the best few...

With that you can get a sense of the epistemic (flow-to-flow) variance on your predictions with something like:
```
for run in range(n_emulators):
    # draw n_posterior samples from the owning model at each point
    posterior_draws = flows[run].make_prediction(context, n_samples=n_samples)
    
    predictions = posterior_draws.mean(axis=0)
    aleatoric   = posterior_draws.std(axis=0)

    # epistemic: disagreement across non-owning models 
    other_preds = np.array([
        flows[other].make_prediction(context, n_samples=n_samples).mean(axis=0)
        for other in range(n_emulators) if other != run
    ]) 

    epistemic = other_preds.std(axis=0)

```